# 1D vs 2D Strain State: Uniaxial and Biaxial Bending

Every solver in this library produces a `StrainState` — an immutable description of how strain varies
across the cross-section. There are two distinct internal solver paths:

| | **1D path** | **2D path** |
|---|---|---|
| Solver | `MNInteractionDiagram` | `FreeNADiagramAdapter` |
| Neutral axis | Horizontal (fixed) | Free to rotate |
| Strain varies with | y only | x and y |
| `StrainState.is_biaxial` | `False` | `True` |
| Key attributes | `eps_top`, `eps_bottom` | `plane_a`, `plane_b`, `plane_c`, `compression_direction` |
| Triggered by | Any `My_Ed`, `N_Ed` | `Mz_Ed ≠ 0` + `free_neutral_axis=True` + asymmetric section |

This notebook shows both paths side-by-side and explains what each `StrainState` field means physically.

## Contents
1. [Setup](#setup)
2. [Build a section](#section)
3. [1D path — uniaxial bending](#1d)
4. [2D path — biaxial bending](#2d)
5. [Visualise the strain planes](#viz)
6. [Strain at arbitrary points](#strain-at)
7. [Effect on code checks — CrackingCheck](#checks)
8. [When is the 2D path activated?](#trigger)

## 1. Setup <a id='setup'></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)
from materials.reinforced_concrete.analysis.interaction_diagram import (
    create_interaction_diagram,
)
from materials.reinforced_concrete.analysis.strain_state import StrainState
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    CrackingCheck,
    LoadDuration,
)

plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK')

## 2. Build a section <a id='section'></a>

We use a **300 × 500 mm rectangular column** with deliberately **asymmetric reinforcement**
about the vertical (minor) axis:
- Bottom: 3 × H25 bars clustered on the left half (x = 40, 100, 160 mm)
- Top: 2 × H16 bars centred symmetrically

The bottom layer is *not* mirrored about the vertical centroidal axis (x ≈ 150 mm),
so `section.is_symmetric_about_vertical_axis()` returns `False`. This is the trigger
that causes `create_interaction_diagram(free_neutral_axis=True)` to return a
`FreeNADiagramAdapter` instead of a plain `MNInteractionDiagram`.

In [ ]:
concrete = ConcreteMaterial(grade='C30/37')
rebar_25 = Rebar(diameter=25)
rebar_16 = Rebar(diameter=16)

width, height = 300, 500   # mm
cover = 40                 # mm to bar centre-line

section = create_rectangular_section(width=width, height=height, section_name='300×500 column')

# Bottom: 3 × H25 clustered on the left half — NOT symmetric about x=150
section.add_rebar_group(create_linear_rebar_layer(
    rebar=rebar_25, n_bars=3,
    start_point=(cover, cover),
    end_point=(160, cover),
    layer_name='btm',
))

# Top: 2 × H16 at symmetric positions
section.add_rebar_group(create_linear_rebar_layer(
    rebar=rebar_16, n_bars=2,
    start_point=(cover, height - cover),
    end_point=(width - cover, height - cover),
    layer_name='top',
))

section.plot(concrete=concrete)

print(f'Section:  {width} × {height} mm')
print(f'Bottom:   3 × H25  (left cluster, x = 40–160 mm)')
print(f'Top:      2 × H16  (symmetric,   x = 40–260 mm)')
print(f'Total As: {section.total_steel_area:.0f} mm²  (ρ = {section.reinforcement_ratio:.3%})')
print()
print(f'Symmetric about vertical axis? {section.is_symmetric_about_vertical_axis()}')
print('(False → FreeNADiagramAdapter will be used when free_neutral_axis=True)')

## 3. 1D path — uniaxial bending (My only) <a id='1d'></a>

The **1D path** uses `MNInteractionDiagram` with a horizontal neutral axis.
Strain varies only along the **y-axis**: `ε(y) = plane_b·y + plane_c`.

Key `StrainState` attributes for the 1D case:
- `is_biaxial = False`
- `plane_a = 0` (no x-dependence)
- `eps_top`, `eps_bottom` — the strains at top and bottom fibres on the centroidal y-axis
- `compression_direction = (0, ±1)` — purely vertical

In [ ]:
# --- 1D solver: standard horizontal-NA diagram ---
diagram_1d = create_interaction_diagram(
    section=section,
    concrete=concrete,
    free_neutral_axis=False,   # 1D path
)

My_Ed = 180.0  # kN·m  (positive = sagging, bottom in tension)
N_Ed  =  50.0  # kN    (compression positive)

eps_top_1d, eps_bottom_1d = diagram_1d.find_strains_for_MN(My_Ed, N_Ed)
ss_1d = diagram_1d.find_strain_state_for_MN(My_Ed, N_Ed)

print('─── 1D StrainState ──────────────────────────────────')
print(f'  is_biaxial         : {ss_1d.is_biaxial}')
print(f'  eps_top            : {ss_1d.eps_top:+.6f}  (compression +)')
print(f'  eps_bottom         : {ss_1d.eps_bottom:+.6f}  (tension –)')
print(f'  plane_a (dε/dx)    : {ss_1d.plane_a:+.6f}  ← zero for horizontal NA')
print(f'  plane_b (dε/dy)    : {ss_1d.plane_b:+.6f}')
print(f'  plane_c (ε at CG)  : {ss_1d.plane_c:+.6f}')
print(f'  compression_dir    : {ss_1d.compression_direction}  ← purely vertical')
print(f'  NA angle (°)       : {ss_1d.get_na_angle_deg():.1f}°  ← horizontal')

In [ ]:
# Cross-check: strain_at() on the centroidal y-axis should match eps_top/eps_bottom
cx = section.get_centroid()[0]  # centroid x (not needed for 1D but shown for clarity)
cy = section.get_centroid()[1]
y_top    = height - cy   # y relative to centroid
y_bottom = 0     - cy

eps_at_top    = ss_1d.strain_at(x=0, y=y_top)     # x=0 → centroidal axis
eps_at_bottom = ss_1d.strain_at(x=0, y=y_bottom)

print('Cross-check: strain_at() vs eps_top / eps_bottom')
print(f'  strain_at(x=0, y_top)    = {eps_at_top:+.6f}   eps_top    = {ss_1d.eps_top:+.6f}')
print(f'  strain_at(x=0, y_bottom) = {eps_at_bottom:+.6f}   eps_bottom = {ss_1d.eps_bottom:+.6f}')
print()
print('For 1D, strain_at() ignores x — the result is the same at any x:')
for x_test in [-100, 0, 100]:
    print(f'  strain_at(x={x_test:+4}, y_top) = {ss_1d.strain_at(x_test - cx + cx, y_top):+.6f}  (identical)')

## 4. 2D path — biaxial bending (My + Mz) <a id='2d'></a>

The **2D path** uses `FreeNADiagramAdapter`, which allows the neutral axis to **rotate**
to satisfy equilibrium in both bending axes simultaneously.
Strain varies with **both x and y**: `ε(x, y) = plane_a·x + plane_b·y + plane_c`.

Triggered when **all three** conditions are met:
1. `Mz_Ed ≠ 0`
2. `free_neutral_axis=True` on the diagram or check
3. Section is asymmetric about the minor (y) axis

Key `StrainState` attributes for the 2D case:
- `is_biaxial = True`
- `plane_a ≠ 0` — strain gradient in x direction due to rotated NA
- `compression_direction` — unit vector pointing toward compression, perpendicular to NA
- `na_angle_deg` — rotation of NA from horizontal

In [ ]:
# --- 2D solver: free neutral axis (biaxial) ---
diagram_2d = create_interaction_diagram(
    section=section,
    concrete=concrete,
    free_neutral_axis=True,   # 2D path — NA can rotate
)

My_Ed = 180.0  # kN·m  (major axis)
Mz_Ed =  40.0  # kN·m  (minor axis — triggers biaxial)
N_Ed  =  50.0  # kN

eps_top_2d, eps_bottom_2d = diagram_2d.find_strains_for_MN(
    My_Ed, N_Ed, Mz_target=Mz_Ed
)
ss_2d = diagram_2d.find_strain_state_for_MN(
    My_Ed, N_Ed, Mz_target=Mz_Ed
)

print('─── 2D StrainState ──────────────────────────────────')
print(f'  is_biaxial         : {ss_2d.is_biaxial}')
print(f'  eps_top            : {ss_2d.eps_top:+.6f}  (at centroidal y-axis, top)')
print(f'  eps_bottom         : {ss_2d.eps_bottom:+.6f}  (at centroidal y-axis, bottom)')
print(f'  plane_a (dε/dx)    : {ss_2d.plane_a:+.6f}  ← NON-ZERO: NA is rotated')
print(f'  plane_b (dε/dy)    : {ss_2d.plane_b:+.6f}')
print(f'  plane_c (ε at CG)  : {ss_2d.plane_c:+.6f}')
print(f'  compression_dir    : ({ss_2d.compression_direction[0]:+.4f}, {ss_2d.compression_direction[1]:+.4f})')
print(f'  NA angle (°)       : {ss_2d.get_na_angle_deg():.2f}°  ← rotated from horizontal')

In [ ]:
# For 2D, strain DOES vary with x — corner strains are all different
cx = section.get_centroid()[0]
cy = section.get_centroid()[1]

corners = {
    'Bottom-left':  (0 - cx,      0 - cy),
    'Bottom-right': (width - cx,  0 - cy),
    'Top-left':     (0 - cx,      height - cy),
    'Top-right':    (width - cx,  height - cy),
}

print('Corner strains (1D vs 2D):')
print(f'  {"Corner":<16} | {"1D ε":>10} | {"2D ε":>10} | {"Difference":>12}')
print('  ' + '-'*55)
for name, (x_rel, y_rel) in corners.items():
    e1d = ss_1d.strain_at(x_rel, y_rel)
    e2d = ss_2d.strain_at(x_rel, y_rel)
    print(f'  {name:<16} | {e1d:+10.6f} | {e2d:+10.6f} | {e2d - e1d:+12.6f}')

print()
print('Note: 1D strains at left and right corners on the SAME row are identical')
print('      2D strains differ because the NA is rotated — x matters too')

## 5. Visualise the strain planes <a id='viz'></a>

The strain field is `ε(x, y) = plane_a·x + plane_b·y + plane_c`.
For 1D, this is a tilted plane with no x-slope — contours are horizontal lines.
For 2D, the plane is tilted in both x and y — contours are diagonal lines matching the neutral axis.

In [ ]:
def plot_strain_field(ax, ss: StrainState, section_w, section_h, cx, cy, title):
    """Heatmap of strain across the section cross-section."""
    xs = np.linspace(0, section_w, 120)
    ys = np.linspace(0, section_h, 120)
    XX, YY = np.meshgrid(xs, ys)
    EE = ss.strain_at(XX - cx, YY - cy)

    vmax = max(abs(EE.min()), abs(EE.max())) or 1e-6
    im = ax.contourf(XX, YY, EE, levels=20, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.contour(XX, YY, EE, levels=[0], colors='k', linewidths=2, linestyles='--')

    # Draw section outline
    rect = patches.Rectangle((0, 0), section_w, section_h,
                               fill=False, edgecolor='k', linewidth=2)
    ax.add_patch(rect)

    ax.set_xlim(-20, section_w + 20)
    ax.set_ylim(-20, section_h + 20)
    ax.set_aspect('equal')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    ax.set_title(title)

    # Add colorbar
    plt.colorbar(im, ax=ax, label='Strain ε (×10⁻³ shown as ×1)',
                 fraction=0.046, pad=0.04)

    # Annotate NA angle
    ang = ss.get_na_angle_deg()
    ax.text(0.02, 0.97, f'NA angle: {ang:.1f}°\nis_biaxial: {ss.is_biaxial}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_strain_field(ax1, ss_1d, width, height, cx, cy,
                 f'1D Strain Field\nMy={My_Ed} kN·m, Mz=0, N={N_Ed} kN')
plot_strain_field(ax2, ss_2d, width, height, cx, cy,
                 f'2D Strain Field\nMy={My_Ed} kN·m, Mz={Mz_Ed} kN·m, N={N_Ed} kN')

fig.suptitle('Strain distribution across section\n(dashed black line = neutral axis)', fontsize=12)
plt.tight_layout()
plt.show()

print('Blue  = tension (negative strain)')
print('Red   = compression (positive strain)')
print('Black dashed = neutral axis (ε = 0)')

## 6. Strain at arbitrary points — `strain_at()` <a id='strain-at'></a>

`StrainState.strain_at(x, y)` evaluates the plane equation at any centroid-relative point.
This is the canonical way to find bar strains inside helper methods.

In [ ]:
# Bar positions (absolute) → centroid-relative
bar_positions_abs = [
    (  cover,          cover,          'btm-L'),
    (  width/2,        cover,          'btm-C'),
    (  width - cover,  cover,          'btm-R'),
    (  cover,          height - cover, 'top-L'),
    (  width - cover,  height - cover, 'top-R'),
]

print(f'Bar strains   (cx = {cx:.1f} mm, cy = {cy:.1f} mm from section origin)')
print(f'  {"Bar":<10} | {"x_rel":>7} | {"y_rel":>7} | {"1D ε":>10} | {"2D ε":>10} | {"Δε":>10}')
print('  ' + '─'*62)

for x_abs, y_abs, name in bar_positions_abs:
    xr = x_abs - cx
    yr = y_abs - cy
    e1 = ss_1d.strain_at(xr, yr)
    e2 = ss_2d.strain_at(xr, yr)
    print(f'  {name:<10} | {xr:+7.1f} | {yr:+7.1f} | {e1:+10.6f} | {e2:+10.6f} | {e2-e1:+10.6f}')

print()
print('Δε = 2D − 1D.  Non-zero Δε is the additional strain from minor-axis bending.')
print('The bottom bars at left vs right have the same 1D strain but different 2D strains.')

In [ ]:
# compression_direction gives the unit vector perpendicular to the NA, toward compression
dx_1d, dy_1d = ss_1d.compression_direction
dx_2d, dy_2d = ss_2d.compression_direction

print('Compression direction vectors:')
print(f'  1D: ({dx_1d:+.4f}, {dy_1d:+.4f})  →  purely upward (top in compression)')
print(f'  2D: ({dx_2d:+.4f}, {dy_2d:+.4f})  →  tilted due to Mz component')
print()
print('Project each bar onto compression_direction to rank bars from tension to compression:')
print(f'  {"Bar":<10} | {"1D proj":>10} | {"2D proj":>10}')
print('  ' + '─'*36)
for x_abs, y_abs, name in bar_positions_abs:
    xr = x_abs - cx
    yr = y_abs - cy
    p1 = ss_1d.project_along_compression(xr, yr)
    p2 = ss_2d.project_along_compression(xr, yr)
    print(f'  {name:<10} | {p1:+10.1f} | {p2:+10.1f}')
print()
print('Larger projection = closer to compression face.  For 1D, left/right bottom bars')
print('project identically.  For 2D they differ because x counts.')

## 7. Effect on code checks — CrackingCheck <a id='checks'></a>

The `CrackingCheck` class supports `free_neutral_axis=True`, which enables the biaxial solver
path when `Mz_Ed ≠ 0`.

Both the 1D and 2D paths now use the **same SLS linear-elastic concrete model**
(`ConcreteModelType.LINEAR_ELASTIC` with `elastic_modulus=E_c,eff`, `include_tension=True`,
`crack_to_neutral_axis_on_first_tension_failure=True`).  `BiaxialMNInteractionSurface` was
extended to accept and forward these parameters, so `FreeNADiagramAdapter` uses the correct
SLS constitutive model.

**Known limitation — biaxial solver convergence at low SLS loads:**
The biaxial equilibrium solver (`scipy.optimize.least_squares`) is a local optimizer and can
find incorrect local minima when loads are far below capacity (small strains, multiple
near-equivalent equilibria).  Results should be verified at load levels where the solver
converges to a physically plausible strain state (bottom in tension for positive `My_Ed`,
`NA angle < 45°`).

| Helper | 1D path | 2D path |
|---|---|---|
| Concrete model | `ConcreteStressStrainLinearElastic(E_c,eff)` | Same — now correctly forwarded |
| Bar strain | `plane_b · y_rel + plane_c` | `plane_a · x_rel + plane_b · y_rel + plane_c` |
| Cover | Y-axis distance to face | Euclidean distance to boundary |
| h_c,ef filter | Distance along y-axis | Projection along `compression_direction` |
| Effective depth | `get_effective_depth(compression_face=...)` | `find_effective_depth_for_flexure(..., strain_state=...)` |

In [ ]:
My_SLS =  90.0   # kN·m
N_SLS  =  30.0   # kN

# 1D cracking check (Mz=0, standard — uses SLS elastic model)
check_1d = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    free_neutral_axis=False,  # 1D path — SLS elastic model (correct)
)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    result_1d = check_1d.perform_check(My_Ed=My_SLS, N_Ed=N_SLS)
cr_1d = result_1d.details

print('─── 1D Cracking Check (Mz = 0) ──────────────────────')
print(f'  w_k          = {cr_1d["w_k"]:.3f} mm')
print(f'  σ_s          = {cr_1d["sigma_s"]:.1f} MPa')
print(f'  s_r,max      = {cr_1d["s_r_max"]:.1f} mm')
print(f'  h_c,ef       = {cr_1d["h_c_ef"]:.1f} mm')
print(f'  Utilization  = {result_1d.utilization:.1%}  [{result_1d.status.value.upper()}]')

In [ ]:
Mz_SLS = 20.0  # kN·m  (minor axis)

# 2D cracking check — executes with biaxial geometry, but note the model limitation above.
# The equilibrium solve uses the ULS nonlinear model, so sigma_s is approximate.
check_2d = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    free_neutral_axis=True,   # biaxial path — ULS model for equilibrium
)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    result_2d = check_2d.perform_check(My_Ed=My_SLS, Mz_Ed=Mz_SLS, N_Ed=N_SLS)
cr_2d = result_2d.details

print('─── 2D Cracking Check (Mz = 20 kN·m) ───────────────')
print(f'  w_k          = {cr_2d["w_k"]:.3f} mm  ← based on ULS model (approximate at SLS)')
print(f'  σ_s          = {cr_2d["sigma_s"]:.1f} MPa')
print(f'  s_r,max      = {cr_2d["s_r_max"]:.1f} mm')
print(f'  h_c,ef       = {cr_2d["h_c_ef"]:.1f} mm')
print(f'  Utilization  = {result_2d.utilization:.1%}  [{result_2d.status.value.upper()}]')
print()
print('NOTE: σ_s may be significantly overestimated because the ULS concrete model')
print('      is used for equilibrium instead of the SLS elastic model (E_c,eff).')

In [ ]:
# Summary: what changes between 1D and 2D geometry helpers
print('What changes with biaxial geometry (when strain_state.is_biaxial = True):')
print()
print('  Cover:')
print(f'    1D (y-axis to face) = {cr_1d["cover"]:.1f} mm')
print(f'    2D (Euclidean dist) = {cr_2d["cover"]:.1f} mm')
print()
print('  h_c,ef:')
print(f'    1D = {cr_1d["h_c_ef"]:.1f} mm  (along y-axis)')
print(f'    2D = {cr_2d["h_c_ef"]:.1f} mm  (projected along compression_direction)')
print()
print('  s_r,max:')
print(f'    1D = {cr_1d["s_r_max"]:.1f} mm  (bars sorted by x-coord or perp to NA)')
print(f'    2D = {cr_2d["s_r_max"]:.1f} mm  (bars sorted perp to compression_direction)')
print()
print('  σ_s (steel stress — note 2D uses approximate ULS model):')
print(f'    1D = {cr_1d["sigma_s"]:.1f} MPa  (SLS elastic — accurate)')
print(f'    2D = {cr_2d["sigma_s"]:.1f} MPa  (ULS model  — approximate at SLS loads)')

### 7.1 Varying My — 1D crack width (Mz = 0)

Since biaxial cracking requires an SLS-compatible biaxial solver (future work), this
sweep uses the 1D path only.  The effect of increasing `My_Ed` on crack width is shown.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

My_values = np.linspace(20, 250, 20)   # kN·m
wk_values = []

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for My in My_values:
        try:
            r = check_1d.perform_check(My_Ed=float(My), N_Ed=N_SLS)
            wk_values.append(r.details['w_k'])
        except Exception:
            wk_values.append(float('nan'))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(My_values, wk_values, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.axhline(0.3, color='red', linestyle='--', linewidth=1.5, label='w_k limit = 0.3 mm')
ax.axvline(My_SLS, color='gray', linestyle=':', linewidth=1.2, label=f'SLS example My = {My_SLS} kN·m')
ax.set_xlabel('Major-axis moment My_Ed (kN·m)', fontsize=11)
ax.set_ylabel('Crack width w_k (mm)', fontsize=11)
ax.set_title(f'Crack width vs My_Ed  (Mz=0, N={N_SLS} kN)\n1D solver path (SLS elastic model)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. When is the 2D path activated? <a id='trigger'></a>

The `create_interaction_diagram(..., free_neutral_axis=True)` factory returns either:

- **`MNInteractionDiagram`** — if the section is **symmetric about the minor axis** (y-axis).
  The rotated-NA solver would produce an identical result to horizontal NA for `Mz = 0`,
  so the cheaper solver is used.
- **`FreeNADiagramAdapter`** — if the section is **asymmetric** about the minor axis.
  The adapter wraps the biaxial solver and accepts `Mz_target` in `find_strains_for_MN`.

At runtime, the `Mz_target` keyword is passed only when `|Mz_Ed| > 1e-9 kN·m`. For `Mz_Ed = 0`,
both solvers are identical — the 2D path collapses to `plane_a = 0` and gives the same `eps_top`
/ `eps_bottom` as the 1D path.

In [ ]:
# Demonstrate: 2D solver with Mz=0 gives same result as 1D solver
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    ss_2d_nomz = diagram_2d.find_strain_state_for_MN(My_Ed, N_Ed)   # No Mz_target

print('2D solver with Mz=0 vs 1D solver:')
print(f'  1D  eps_top    = {ss_1d.eps_top:+.8f}   is_biaxial = {ss_1d.is_biaxial}')
print(f'  2D  eps_top    = {ss_2d_nomz.eps_top:+.8f}   is_biaxial = {ss_2d_nomz.is_biaxial}')
print(f'  1D  eps_bottom = {ss_1d.eps_bottom:+.8f}')
print(f'  2D  eps_bottom = {ss_2d_nomz.eps_bottom:+.8f}')
print(f'  Difference     = {abs(ss_1d.eps_top - ss_2d_nomz.eps_top):.2e} (numerical solver tolerance)')

print()
print('Trigger summary:')
print('  |Mz_Ed| = 0                 → 1D path (plane_a = 0, eps_top/eps_bottom valid)')
print('  |Mz_Ed| > 0, free_NA=False  → ValueError (Mz_Ed requires free_neutral_axis=True)')
print('  |Mz_Ed| > 0, free_NA=True,  → 2D path (plane_a ≠ 0, full plane equation)')
print('    asymmetric section')
print('  |Mz_Ed| > 0, free_NA=True,  → 2D path collapses back to 1D (plane_a ≈ 0)')
print('    symmetric section          (section is symmetric about minor axis)')

In [ ]:
# Confirm: symmetric section with free_NA gives is_biaxial=False even with Mz≠0
sym_section = create_rectangular_section(width=300, height=500, section_name='sym')
for y_pos in [cover, height - cover]:
    sym_section.add_rebar_group(create_linear_rebar_layer(
        rebar=rebar_25, n_bars=3,
        start_point=(cover, y_pos),
        end_point=(width - cover, y_pos),
    ))

diag_sym = create_interaction_diagram(section=sym_section, concrete=concrete, free_neutral_axis=True)

print(f'Symmetric section, free_NA=True, type = {type(diag_sym).__name__}')
print('(Returns MNInteractionDiagram — symmetric section detected, 2D adapter not needed)')
print()

diag_asym = create_interaction_diagram(section=section, concrete=concrete, free_neutral_axis=True)
print(f'Asymmetric section, free_NA=True, type = {type(diag_asym).__name__}')
print('(Returns FreeNADiagramAdapter — accepts Mz_target in find_strains_for_MN)')

## Summary

| | **1D path** | **2D path** |
|---|---|---|
| Class | `MNInteractionDiagram` | `FreeNADiagramAdapter` |
| `StrainState.is_biaxial` | `False` | `True` |
| `plane_a` | `0` | `≠ 0` |
| Strain at bar | `plane_b * y_rel + plane_c` | `plane_a * x_rel + plane_b * y_rel + plane_c` |
| `compression_direction` | `(0, ±1)` | Tilted unit vector |
| Concrete model (SLS) | `ConcreteStressStrainLinearElastic(E_c,eff)` | Same — forwarded correctly |
| Cover (cracking) | Y-axis distance to face | Euclidean distance to boundary |
| h_c,ef filter | Distance along y-axis | Projection along `compression_direction` |
| Effective depth | `get_effective_depth(compression_face=...)` | `find_effective_depth_for_flexure(..., strain_state=...)` |

### Solver convergence note
The biaxial solver (`scipy.optimize.least_squares`) can find wrong local minima at low SLS
load levels.  Always verify that `strain_state.eps_bottom < 0 < strain_state.eps_top` for
positive `My_Ed` and that `get_na_angle_deg()` is physically plausible.  This is a
solver-algorithm limitation and is separate from the constitutive model implementation, which
is now correct for both paths.

The two paths are completely separate inside every geometry helper — 2D never falls back to
Y-axis face terminology, and 1D never calls biaxial helpers.